In [4]:
!pip install groq duckduckgo-search pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 121.3 MB/s eta 0:00:00


In [5]:

import os
os.environ["GROQ_API_KEY"] = "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"


In [13]:
from groq import Groq
client = Groq()

def call_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [14]:
from duckduckgo_search import DDGS

def web_search(query):
    try:
        with DDGS() as ddgs:
            results = [r["body"] for r in ddgs.text(query, max_results=3)]
        return "\n".join(results)
    except Exception as e:
        return f"Search Error: {str(e)}"

In [15]:
import traceback

def run_python(code):
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        return str(local_vars)
    except Exception as e:
        return f"Execution Error: {traceback.format_exc()}"

In [16]:
def write_file(input_text):
    try:
        parts = input_text.split("||")
        filename = parts[0]
        content = parts[1]

        with open(filename, "w") as f:
            f.write(content)

        return f"Saved to {filename}"
    except Exception as e:
        return f"Write Error: {str(e)}"

In [17]:
class AutonomousAgent:
    def __init__(self):
        self.tools = {
            "search": web_search,
            "python": run_python,
            "write": write_file
        }
        self.logs = []

    def log(self, msg):
        print(msg)
        self.logs.append(msg)

    # 🧠 Planning
    def plan(self, task):
        prompt = f"""
        Break the task into clear numbered steps.
        Task: {task}
        """
        return call_llm(prompt)

    # 🤔 Decide tool
    def decide(self, step):
        prompt = f"""
        You are an agent.

        Available tools:
        - search(query)
        - python(code)
        - write(filename||content)

        Step: {step}

        Return EXACT format:
        TOOL: tool_name
        INPUT: input_here
        """
        return call_llm(prompt)

    # ⚙️ Execute tool
    def execute(self, tool, tool_input):
        if tool not in self.tools:
            return "Error: Invalid Tool"
        return self.tools[tool](tool_input)

    # 🔁 Solve with Replanning
    def solve(self, task):
        self.log(f"\n===== TASK: {task} =====")

        plan = self.plan(task)
        self.log(f"\nPLAN:\n{plan}")

        steps = plan.split("\n")
        outputs = []

        for step in steps:
            if not step.strip():
                continue

            self.log(f"\nSTEP: {step}")

            decision = self.decide(step)
            self.log(f"DECISION:\n{decision}")

            # Parse safely
            try:
                tool = decision.split("TOOL:")[1].split("\n")[0].strip()
                tool_input = decision.split("INPUT:")[1].strip()
            except:
                self.log("⚠️ Parsing failed → Skipping step")
                continue

            result = self.execute(tool, tool_input)
            self.log(f"OUTPUT:\n{result}")

            # 🔁 Dynamic Replanning
            if "Error" in result:
                self.log("⚠️ Error detected → Replanning step")
                retry_decision = self.decide(step + " (retry with different approach)")

                try:
                    tool = retry_decision.split("TOOL:")[1].split("\n")[0].strip()
                    tool_input = retry_decision.split("INPUT:")[1].strip()
                    result = self.execute(tool, tool_input)
                    self.log(f"RETRY OUTPUT:\n{result}")
                except:
                    self.log("❌ Retry failed")

            outputs.append(result)

        return outputs

In [11]:
tasks = [
    "Find latest AI trends and summarize",
    "Search Tesla stock trend and summarize",
    "Write python code to plot sine wave",
    "Explain quantum computing in simple words",
    "Find top 3 programming languages in 2025",
    "Generate python code for fibonacci series",
    "Search climate change impacts",
    "Write a short blog on AI agents",
    "Analyze benefits of exercise",
    "Save a file named report.txt with AI summary"
]


In [18]:
agent = AutonomousAgent()

results = []
for task in tasks:
    res = agent.solve(task)
    results.append(res)

print("\n✅ ALL TASKS COMPLETED")


===== TASK: Find latest AI trends and summarize =====

PLAN:
Here are the steps to find the latest AI trends and summarize them:

**Step 1: Conduct Research**

1.1. Browse through reputable AI-related websites, blogs, and online publications (e.g. MIT Technology Review, TechCrunch, AI Research News)
1.2. Check social media platforms (e.g. Twitter, LinkedIn) for AI-related discussions and insights
1.3. Search academic journals and research papers (e.g. arXiv, Semantic Scholar) for the latest AI research and breakthroughs

**Step 2: Identify Recent AI Conferences and Events**

2.1. Look for upcoming and recent AI conferences (e.g. NeurIPS, ICLR, ICML)
2.2. Review the conference agendas and keynote speeches
2.3. Follow live coverage and summaries from industry leaders and researchers on social media and blogs

**Step 3: Gather Information on Latest AI Models and Techniques**

3.1. Identify the latest pre-trained models and their applications (e.g. DALL-E, GPT-3)
3.2. Investigate new AI t

/tmp/ipykernel_464/3287366053.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


RETRY OUTPUT:


STEP: **Step 1: Conduct Research**
DECISION:
TOOL: search()
INPUT: cyber security threats and countermeasures
OUTPUT:
Error: Invalid Tool
⚠️ Error detected → Replanning step


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


RETRY OUTPUT:
Error: Invalid Tool

STEP: 1.1. Browse through reputable AI-related websites, blogs, and online publications (e.g. MIT Technology Review, TechCrunch, AI Research News)
DECISION:
TOOL: search(query)
INPUT: "Artificial intelligence latest news" "AI updates 2024"
OUTPUT:
Error: Invalid Tool
⚠️ Error detected → Replanning step


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

DECISION:
TOOL: search(query)
INPUT: "cross-check information gathered from multiple sources"

TOOL: write(filename||content)
INPUT: "information_gathering.txt"
CONTENT: 
"Cross-checking information gathered from multiple sources involves verifying the accuracy and consistency of data collected from different sources. 

1. Compare and contrast the information gathered from each source.
2. Identify and address any discrepancies or inconsistencies found.
3. Use a checklist or a matrix to organize the information and make it easier to compare.
4. Consider using a data validation tool to automatically check for errors and inconsistencies.
5. Use a decision matrix to evaluate the reliability of each source and prioritize the sources accordingly.

By following these steps, you can ensure that the information gathered from multiple sources is accurate, reliable, and trustworthy."

TOOL: search(query)
INPUT: "decision matrix"

TOOL: python(code)
INPUT: 
```
# Define a dictionary to store the d

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kq73b0qffdxs1m7ncnc6mv1g` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5579, Requested 436. Please try again in 150ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [19]:
success = sum(1 for r in results if len(r) > 0)

print("Total Tasks:", len(tasks))
print("Successful:", success)
print("Success Rate:", success / len(tasks))

Total Tasks: 10
Successful: 0
Success Rate: 0.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [20]:
with open("agent_logs.txt", "w") as f:
    for log in agent.logs:
        f.write(log + "\n")

print("Logs saved!")

Logs saved!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag